# BUILDING MUSIC RECOMMENDER

# Importing Modules

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import time
import sklearn
from sklearn.model_selection import train_test_split
import Recommenders as Recommenders
import Evaluation as Evaluation

# Loading Datasets

In [2]:
#loading first dataset named as triplet file which consist of song ID, User ID, Listen count
song_df_1 = pd.read_csv('triplets_file.csv')
song_df_1.head(11)

,user_id,song_id,listen_count
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1
1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBBMDR12A8C13253B,2
2,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBXHDL12A81C204C0,1
3,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBYHAJ12A6701BF1D,1
4,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SODACBL12A8C13C273,1
5,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SODDNQT12A6D4F5F7E,5
6,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SODXRTY12AB0180F3B,1
7,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOFGUAY12AB017B0A8,1
8,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOFRQTD12A81C233C0,1
9,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOHQWYZ12A6D4FA701,1


In [3]:
#we read the second dataset(song data) which have song ID,artists, release yr, title in it
song_df_2 = pd.read_csv('song_data.csv')
song_df_2.head(11)

,song_id,title,release,artist_name,year
0,SOQMMHC12AB0180CB8,Silent Night,Monster Ballads X-Mas,Faster Pussy cat,2003
1,SOVFVAK12A8C1350D9,Tanssi vaan,Karkuteillä,Karkkiautomaatti,1995
2,SOGTUKN12AB017F4F1,No One Could Ever,Butter,Hudson Mohawke,2006
3,SOBNYVR12A8C13558C,Si Vos Querés,De Culo,Yerba Brava,2003
4,SOHSBXH12A8C13B0DF,Tangle Of Aspens,Rene Ablaze Presents Winter Sessions,Der Mystic,0
5,SOZVAPQ12A8C13B63C,"Symphony No. 1 G minor ""Sinfonie Serieuse""/All...",Berwald: Symphonies Nos. 1/2/3/4,David Montgomery,0
6,SOQVRHI12A6D4FB2D7,We Have Got Love,Strictly The Best Vol. 34,Sasha / Turbulence,0
7,SOEYRFT12AB018936C,2 Da Beat Ch'yall,Da Bomb,Kris Kross,1993
8,SOPMIYT12A6D4F851E,Goodbye,Danny Boy,Joseph Locke,0
9,SOJCFMH12A8C13B0C2,Mama_ mama can't you see ?,March to cadence with the US marines,The Sun Harbor's Chorus-Documentary Recordings,0


# Preprocessing Data

In [4]:
## combine data of titel and artists name with help of Song ID as it is common in both
#by drop_duplicates we just removed the copies of sings in dataset
song_df = pd.merge(song_df_1, song_df_2.drop_duplicates(['song_id']), on='song_id')
song_df.head(11)

,user_id,song_id,listen_count,title,release,artist_name,year
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0
1,7c86176941718984fed11b7c0674ff04c029b480,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0
2,76235885b32c4e8c82760c340dc54f9b608d7d7e,SOAKIMP12A8C130995,3,The Cove,Thicker Than Water,Jack Johnson,0
3,250c0fa2a77bc6695046e7c47882ecd85c42d748,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0
4,3f73f44560e822344b0fb7c6b463869743eb9860,SOAKIMP12A8C130995,6,The Cove,Thicker Than Water,Jack Johnson,0
5,7a4b8e7d2905d13422418b4f48cc85100892e013,SOAKIMP12A8C130995,6,The Cove,Thicker Than Water,Jack Johnson,0
6,b4a678fb729bfca6031a96948996ea909ca06fe5,SOAKIMP12A8C130995,2,The Cove,Thicker Than Water,Jack Johnson,0
7,33280fc74b168e2667a2da5c6ab4df4cc6edfb23,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0
8,be21ec120193effd2a5e545c4bafa2e1f92e9816,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0
9,6fbb9ff93663f3c3ad206a9325d90b19278618b4,SOAKIMP12A8C130995,2,The Cove,Thicker Than Water,Jack Johnson,0


Creating a SUBSET of Dataset

In [5]:
# creating new feature by combining title and artist name
song_df['song'] = song_df['title']+' - '+song_df['artist_name']
song_df.head()

,user_id,song_id,listen_count,title,release,artist_name,year,song
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
1,7c86176941718984fed11b7c0674ff04c029b480,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
2,76235885b32c4e8c82760c340dc54f9b608d7d7e,SOAKIMP12A8C130995,3,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
3,250c0fa2a77bc6695046e7c47882ecd85c42d748,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
4,3f73f44560e822344b0fb7c6b463869743eb9860,SOAKIMP12A8C130995,6,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson


In [6]:
#dropping unessential columns of the data frame
song_df.drop(['artist_name','title'],axis=1)


,user_id,song_id,listen_count,release,year,song
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1,Thicker Than Water,0,The Cove - Jack Johnson
1,7c86176941718984fed11b7c0674ff04c029b480,SOAKIMP12A8C130995,1,Thicker Than Water,0,The Cove - Jack Johnson
2,76235885b32c4e8c82760c340dc54f9b608d7d7e,SOAKIMP12A8C130995,3,Thicker Than Water,0,The Cove - Jack Johnson
3,250c0fa2a77bc6695046e7c47882ecd85c42d748,SOAKIMP12A8C130995,1,Thicker Than Water,0,The Cove - Jack Johnson
4,3f73f44560e822344b0fb7c6b463869743eb9860,SOAKIMP12A8C130995,6,Thicker Than Water,0,The Cove - Jack Johnson
...,...,...,...,...,...,...
1048570,aba4d5d8061afda8df24e5d9320b71299ca392e0,SOPBPHJ12AAF3B59B6,1,Crazy Love,2009,Baby [You've Got What It Takes] [with Sharon J...
1048571,93506a4788c168b273beae4ba52a1d592f9b8e9e,SOPBPHJ12AAF3B59B6,2,Crazy Love,2009,Baby [You've Got What It Takes] [with Sharon J...
1048572,cdf8d4253caf64cc1f3b4e5ae1b038735efdfd91,SOPBPHJ12AAF3B59B6,2,Crazy Love,2009,Baby [You've Got What It Takes] [with Sharon J...
1048573,a7e3dde43d20adaf50e1fe56fb9abb4ba6e2bed7,SOPBPHJ12AAF3B59B6,1,Crazy Love,2009,Baby [You've Got What It Takes] [with Sharon J...


LENGTH of Dataset

In [7]:
len(song_df)

1048575

Count number of unique USERS in the dataset

In [8]:
users = song_df['user_id'].unique()
len(users) 

40336

Count number of unique SONGS in the dataset

In [9]:
songs = song_df['song'].unique()
len(songs)

9953

# EXPLORING THE DATASET

The MOST PLAYED SONGS in dataset are

In [10]:
# taking top 10k samples for quick results as when we tried to process the whole dataset it took around 10-15 minutes, so for better and fast way of getting reults we reduced the size

song_df = song_df.head(100000)

In [26]:
song_grouped = song_df.groupby(['song']).agg({'listen_count': 'count'}).reset_index()
grouped_sum = song_grouped['listen_count'].sum()
song_grouped['percentage']  = song_grouped['listen_count'].div(grouped_sum)*100
song_grouped.sort_values(['listen_count'],ascending=False)

,song,listen_count,percentage
200,Sehr kosmisch - Harmonia,4368,4.368
267,Undo - Björk,3693,3.693
63,Dog Days Are Over (Radio Edit) - Florence + Th...,3676,3.676
288,You're The One - Dwight Yoakam,3336,3.336
187,Revelry - Kings Of Leon,3179,3.179
...,...,...,...
203,She'll Come Back To Me - Cake,37,0.037
109,High and dry - Jorge Drexler,37,0.037
50,Corn Bread - DAVE MATTHEWS BAND,34,0.034
171,Oh! - Sleater-kinney,34,0.034


Showing the DETAILS of songs an User have played

In [12]:
song_df

,user_id,song_id,listen_count,title,release,artist_name,year,song
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
1,7c86176941718984fed11b7c0674ff04c029b480,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
2,76235885b32c4e8c82760c340dc54f9b608d7d7e,SOAKIMP12A8C130995,3,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
3,250c0fa2a77bc6695046e7c47882ecd85c42d748,SOAKIMP12A8C130995,1,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
4,3f73f44560e822344b0fb7c6b463869743eb9860,SOAKIMP12A8C130995,6,The Cove,Thicker Than Water,Jack Johnson,0,The Cove - Jack Johnson
...,...,...,...,...,...,...,...,...
99995,8d9aea5e7f5e5d4302d1266a1c172a1de99be594,SOYDTIW12A67ADAFC9,1,The Police And The Private,Live It Out,Metric,2005,The Police And The Private - Metric
99996,31715ba950602bff5cb8a8d7b8d8ae10e06f0499,SOYDTIW12A67ADAFC9,2,The Police And The Private,Live It Out,Metric,2005,The Police And The Private - Metric
99997,dfe0191c5a9a250e32c8cc9f0d5f247454268122,SOYDTIW12A67ADAFC9,2,The Police And The Private,Live It Out,Metric,2005,The Police And The Private - Metric
99998,e8a7e3dc2265ab63964297841a177e2074100c52,SOYDTIW12A67ADAFC9,1,The Police And The Private,Live It Out,Metric,2005,The Police And The Private - Metric


# CREATING A SONG RECOMMENDER

In [13]:
#creating a song recommender by splitting our dataset into training and testing data
#It’s important to note that whenever we build a machine learning system, before we train our model, we always want to split our data into training and testing dataset

#we keep 20% of dataset aside for testing and the dataset below is from train_data
train_data, test_data = train_test_split(song_df, test_size = 0.20, random_state=0)
train_data

,user_id,song_id,listen_count,title,release,artist_name,year,song
10382,dbff660e89a235c2599b284e0a09523952027a04,SOTFATN12A6D4FA74D,5,The Christmas Song (LP Version),Songs You Know - Christmas Soul Classics,King Curtis,0,The Christmas Song (LP Version) - King Curtis
73171,0c9bf50fff17c2a0ea9c930a7e4cb42b50ccfe66,SOBJIZY12A6701F11A,2,Emotion,Human After All,Daft Punk,2005,Emotion - Daft Punk
30938,6d7f8fffcbe7e1f82520f7b016444d7fe62ad385,SOAXGDH12A8C13F8A1,1,Dog Days Are Over (Radio Edit),Now That's What I Call Music! 75,Florence + The Machine,0,Dog Days Are Over (Radio Edit) - Florence + Th...
99310,79bff7dfecd3382ddf002c7fbcc64f77d2cc1547,SOVZOBS12A67ADAFC6,2,Too Little Too Late,Live It Out,Metric,2005,Too Little Too Late - Metric
58959,5570c7f13f03cbf5bade2656281ef431b2c63af0,SOVDSJC12A58A7A271,10,Ain't Misbehavin,Summertime,Sam Cooke,0,Ain't Misbehavin - Sam Cooke
...,...,...,...,...,...,...,...,...
21243,689fafec535fafda718d0ccc8957bd193984cb19,SOEPZQS12A8C1436C7,1,Ghosts 'n' Stuff (Original Instrumental Mix),Ghosts 'n' Stuff,Deadmau5,2009,Ghosts 'n' Stuff (Original Instrumental Mix) -...
45891,382923fa6a0a93f79823e183372738600d5a3a11,SONYKOW12AB01849C9,1,Secrets,Waking Up,OneRepublic,2009,Secrets - OneRepublic
42613,e224ba26339c840a564a95f263d2a8a4903cb5a8,SOEGIYH12A6D4FC0E3,1,Horn Concerto No. 4 in E flat K495: II. Romanc...,Mozart - Eine kleine Nachtmusik,Barry Tuckwell/Academy of St Martin-in-the-Fie...,0,Horn Concerto No. 4 in E flat K495: II. Romanc...
43567,3ece03154fe68f2f4dab108e0978b7ca9b8697d4,SOLFXKT12AB017E3E0,2,Fireflies,Karaoke Monthly Vol. 2 (January 2010),Charttraxx Karaoke,2009,Fireflies - Charttraxx Karaoke


In [14]:
len(train_data)

80000

In [15]:
test_data

,user_id,song_id,listen_count,title,release,artist_name,year,song
3582,72ee105e7cc43ad166c9023c94ed5f3a0c355da9,SOFRQTD12A81C233C0,4,Sehr kosmisch,Musik von Harmonia,Harmonia,0,Sehr kosmisch - Harmonia
60498,18362253b54f5705115b98bed5fd6a00445c0467,SOWCKVR12A8C142411,1,Use Somebody,Use Somebody,Kings Of Leon,2008,Use Somebody - Kings Of Leon
53227,9ba5dd7d673b76224b6f0f28c1868488303b174f,SOTWNDJ12A8C143984,4,Marry Me,Save Me_ San Francisco,Train,2009,Marry Me - Train
21333,74d965061c841c6271fdc7d025055816f68bb257,SOGDDKR12A6701E8FA,1,My Dad's Gone Crazy,The Eminem Show,Eminem / Hailie Jade,2006,My Dad's Gone Crazy - Eminem / Hailie Jade
3885,122582f3c54f78d5c52fbdeb1065720922a6276f,SOFRQTD12A81C233C0,10,Sehr kosmisch,Musik von Harmonia,Harmonia,0,Sehr kosmisch - Harmonia
...,...,...,...,...,...,...,...,...
60116,adb4bb9b129a2ed96d4214955a163be64e5c7aa7,SOWCKVR12A8C142411,1,Use Somebody,Use Somebody,Kings Of Leon,2008,Use Somebody - Kings Of Leon
2415,bb61c13e5bad026774a16375b70030f01b248f12,SOFRQTD12A81C233C0,2,Sehr kosmisch,Musik von Harmonia,Harmonia,0,Sehr kosmisch - Harmonia
43763,d3057d321d0be431cde08f35f0a542719628336c,SOLFXKT12AB017E3E0,7,Fireflies,Karaoke Monthly Vol. 2 (January 2010),Charttraxx Karaoke,2009,Fireflies - Charttraxx Karaoke
71345,cf21732c086b49edbde9f876aa1d8a4ffd6869c3,SOVFDZD12A6D4F8EAE,2,Raining Again (Steve Angello's Vocal Mix),Hotel: The Remixes,Moby,2005,Raining Again (Steve Angello's Vocal Mix) - Moby


In [16]:
len(test_data)

20000

# Popularity based recommender

In [17]:
#working of recommender
#Get a count of user_ids for each unique song as recommendation score(score is given by the number of time that song was listened to in general by all use)
#Sort the songs based upon recommendation score
#Generate a recommendation rank based upon score
#Get the top 10 recommendationspm = Recommenders.popularity_recommender_py()

Create an instance of popularity based recommender class 

In [18]:
pm = Recommenders.popularity_recommender_py()
pm.create(train_data, 'user_id', 'song')

Use the popularity model to make some predictions

In [19]:
#prediction for user 5
user_id = users[5]
pm.recommend(user_id)

,user_id,song,score,Rank
200,7a4b8e7d2905d13422418b4f48cc85100892e013,Sehr kosmisch - Harmonia,3499,1.0
63,7a4b8e7d2905d13422418b4f48cc85100892e013,Dog Days Are Over (Radio Edit) - Florence + Th...,2952,2.0
267,7a4b8e7d2905d13422418b4f48cc85100892e013,Undo - Björk,2951,3.0
288,7a4b8e7d2905d13422418b4f48cc85100892e013,You're The One - Dwight Yoakam,2655,4.0
187,7a4b8e7d2905d13422418b4f48cc85100892e013,Revelry - Kings Of Leon,2561,5.0
199,7a4b8e7d2905d13422418b4f48cc85100892e013,Secrets - OneRepublic,2401,6.0
111,7a4b8e7d2905d13422418b4f48cc85100892e013,Horn Concerto No. 4 in E flat K495: II. Romanc...,2274,7.0
82,7a4b8e7d2905d13422418b4f48cc85100892e013,Fireflies - Charttraxx Karaoke,2005,8.0
107,7a4b8e7d2905d13422418b4f48cc85100892e013,Hey_ Soul Sister - Train,1984,9.0
268,7a4b8e7d2905d13422418b4f48cc85100892e013,Use Somebody - Kings Of Leon,1639,10.0


In [28]:
#prediction for user 120
user_id = users[120]
pm.recommend(user_id)

,user_id,song,score,Rank
200,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Sehr kosmisch - Harmonia,3499,1.0
63,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Dog Days Are Over (Radio Edit) - Florence + Th...,2952,2.0
267,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Undo - Björk,2951,3.0
288,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,You're The One - Dwight Yoakam,2655,4.0
187,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Revelry - Kings Of Leon,2561,5.0
199,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Secrets - OneRepublic,2401,6.0
111,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Horn Concerto No. 4 in E flat K495: II. Romanc...,2274,7.0
82,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Fireflies - Charttraxx Karaoke,2005,8.0
107,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Hey_ Soul Sister - Train,1984,9.0
268,f0ba0e03b2d41d60ae449a3989d8ee961afe3e94,Use Somebody - Kings Of Leon,1639,10.0


# Similar item based recommender

It is for personalized recommendation for a user, based on the song user played, what genres user likes and etc

Create an instance of item similarity based recommender class

In [21]:
is_model = Recommenders.item_similarity_recommender_py()
is_model.create(train_data, 'user_id', 'song')


In [32]:
#Print the songs for the user in training data
user_id = users[200]
user_items = is_model.get_user_items(user_id)
#
print("------------------------------------------------------------------------------------")
print("Training data songs for the user userid: %s:" % user_id)
print("------------------------------------------------------------------------------------")

for user_item in user_items:
    print(user_item)

print("----------------------------------------------------------------------")
print("Recommendation process going on:")
print("----------------------------------------------------------------------")

#Recommend songs for the user using personalized model
is_model.recommend(user_id)

------------------------------------------------------------------------------------
Training data songs for the user userid: 93e4c342bc05f215b77a225054c403f6ea5a9fab:
------------------------------------------------------------------------------------
Harder Better Faster Stronger - Daft Punk
Drop The World - Lil Wayne / Eminem
Dog Days Are Over (Radio Edit) - Florence + The Machine
Bulletproof - La Roux
Stronger - Kanye West
OMG - Usher featuring will.i.am
Fireflies - Charttraxx Karaoke
----------------------------------------------------------------------
Recommendation process going on:
----------------------------------------------------------------------
No. of unique songs for the user: 7
no. of unique songs in the training set: 289
Non zero values in cooccurence_matrix :1884


,user_id,song,score,rank
0,93e4c342bc05f215b77a225054c403f6ea5a9fab,Secrets - OneRepublic,0.132936,1
1,93e4c342bc05f215b77a225054c403f6ea5a9fab,Hey_ Soul Sister - Train,0.122519,2
2,93e4c342bc05f215b77a225054c403f6ea5a9fab,Marry Me - Train,0.118068,3
3,93e4c342bc05f215b77a225054c403f6ea5a9fab,Sehr kosmisch - Harmonia,0.117727,4
4,93e4c342bc05f215b77a225054c403f6ea5a9fab,Use Somebody - Kings Of Leon,0.106906,5
5,93e4c342bc05f215b77a225054c403f6ea5a9fab,Somebody To Love - Justin Bieber,0.096187,6
6,93e4c342bc05f215b77a225054c403f6ea5a9fab,The Scientist - Coldplay,0.091625,7
7,93e4c342bc05f215b77a225054c403f6ea5a9fab,Heartbreak Warfare - John Mayer,0.080015,8
8,93e4c342bc05f215b77a225054c403f6ea5a9fab,Clocks - Coldplay,0.076999,9
9,93e4c342bc05f215b77a225054c403f6ea5a9fab,Revelry - Kings Of Leon,0.073758,10


We can also find songs that will be recommended based on similar song types/ words

In [33]:
song = 'Yellow - Coldplay'
##Fill in the code here
is_model.get_similar_items([song])

no. of unique songs in the training set: 289
Non zero values in cooccurence_matrix :276


,user_id,song,score,rank
0,,The Scientist - Coldplay,0.200466,1
1,,Clocks - Coldplay,0.184578,2
2,,In My Place - Coldplay,0.144603,3
3,,Speed Of Sound - Coldplay,0.130146,4
4,,Use Somebody - Kings Of Leon,0.091331,5
5,,Hey_ Soul Sister - Train,0.082792,6
6,,Secrets - OneRepublic,0.081848,7
7,,Heartbreak Warfare - John Mayer,0.078921,8
8,,Marry Me - Train,0.076400,9
9,,Life In Technicolor ii - Coldplay,0.076226,10


# Code to plot precision recall curve

In [24]:
import pylab as pl

#Method to generate precision and recall curve
def plot_precision_recall(m1_precision_list, m1_recall_list, m1_label, m2_precision_list, m2_recall_list, m2_label):
    pl.clf()    
    pl.plot(m1_recall_list, m1_precision_list, label=m1_label)
    pl.plot(m2_recall_list, m2_precision_list, label=m2_label)
    pl.xlabel('Recall')
    pl.ylabel('Precision')
    pl.ylim([0.0, 0.20])
    pl.xlim([0.0, 0.20])
    pl.title('Precision-Recall curve')
    #pl.legend(loc="upper right")
    pl.legend(loc=9, bbox_to_anchor=(0.5, -0.2))
    pl.show()


In [25]:
print("Plotting precision recall curves.")

plot_precision_recall(pm_avg_precision_list, pm_avg_recall_list, "popularity_model",
                      ism_avg_precision_list, ism_avg_recall_list, "item_similarity_model")

Plotting precision recall curves.


NameError: name 'pm_avg_precision_list' is not defined